# Weighbridge Routing of the Optimal Paddy-Logistics Solution

This notebook takes the **optimal road flows** produced by the distribution model
(`mycode_final_completed.ipynb`) and routes every road movement through a **weighbridge (WB)**,
exactly as described in **Section 4.3** of the Mid-Term Report.

**Rule (Section 4.3).** A road consignment from node `p` to node `q` is not shipped on the direct
path; it must be weighed at a weighbridge, so the realised route is a *two-leg* path `p -> wb* -> q`:

1. `wb1` = weighbridge nearest to the **origin** `p`  &nbsp;→&nbsp; `argmin_b d(p, b)`
2. `wb2` = weighbridge nearest to the **destination** `q` &nbsp;→&nbsp; `argmin_b d(q, b)`
3. `L1 = d(p, wb1) + d(wb1, q)`  and  `L2 = d(p, wb2) + d(wb2, q)`
4. choose the **shorter** of the two:  `d̃(p,q) = min(L1, L2)`, and `wb*` is the corresponding weighbridge
5. the flow `p -> q` is reported as **two legs** `p -> wb*` and `wb* -> q`, each carrying the same quantity

By the triangle inequality `d̃(p,q) ≥ d(p,q)`, so weighbridge routing can only *add* distance (the detour).

**Input.** This notebook reads `optimal_flows.csv` (columns `tag, i, j, flow`) — the model's road flows.
Produce it by adding this one cell at the end of the model notebook, **after** it has solved:

```python
# export the solved flows for the weighbridge notebook
import pandas as pd
rows = [(t, i, j, (v.varValue or 0.0))
        for t, dd in Allocation.items() for (i, j), v in dd.items() if (v.varValue or 0.0) > 1e-6]
pd.DataFrame(rows, columns=['tag','i','j','flow']).to_csv('optimal_flows.csv', index=False)
# (for the rail far-pairs, also:)
pd.DataFrame([(i, j, RailVia[(i, j)]) for (i, j) in Allocation['PCS_SPD_RAIL']],
             columns=['i','j','r']).to_csv('railvia.csv', index=False)
pd.DataFrame([(int(x.i), int(x.j), float(x.share)) for x in usable.itertuples()],
             columns=['r','s','share']).to_csv('rail_shares.csv', index=False)
```


In [ ]:
import numpy as np
import pandas as pd

# ---------------- load the SAME node tables as the model (row order defines the i / j indices) ----------------
xl   = pd.ExcelFile('Backend//Data_11.xlsx')
PC   = pd.read_excel(xl, 'A.1 Warehouse')   # procurement centres
Mill = pd.read_excel(xl, 'A.2 FPS')         # mills
WH   = pd.read_excel(xl, 'A.2 Warehouse')   # warehouses (state warehouses, SW)
SP   = pd.read_excel(xl, 'A.3 Storage')     # storage points
RH   = pd.read_excel(xl, 'A.3 RH')          # rail heads
WB   = pd.read_excel(xl, 'Weighbridge')     # weighbridges
print('weighbridges:', len(WB), '| columns:', list(WB.columns))

In [ ]:
# ---------------- distance + per-node-type lookup ----------------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# node type -> (dataframe, lat col, lon col, id col, name col)
NODETYPE = {
    'PC'  : (PC,   'WH_Lat', 'WH_Long', 'WH_ID',  'WH_Name'),
    'Mill': (Mill, 'FPS_Lat','FPS_Long','FPS_ID', 'FPS_Name'),
    'SP'  : (SP,   'WH_Lat', 'WH_Long', 'WH_ID',  'WH_Name'),
    'WH'  : (WH,   'SW_Lat', 'SW_Long', 'SW_ID',  'SW_Name'),
    'RH'  : (RH,   'WH_Lat', 'WH_Long', 'WH_ID',  'WH_Name'),
}
# each ROAD tag -> (origin node type, destination node type)
TAG_ENDS = {
    'PCS_MILLS': ('PC','Mill'), 'PCS_SPS': ('PC','SP'),  'SPS_MILLS': ('SP','Mill'),
    'MILLS_SWS': ('Mill','WH'), 'PCS_RHS': ('PC','RH'),  'PCS_SPD':  ('PC','SP'),
    'PCD_SPD':   ('PC','SP'),   'PCD_MILLD': ('PC','Mill'), 'SPD_MILLD': ('SP','Mill'),
    'RHD_MILLD': ('RH','Mill'), 'MILLD_SWD': ('Mill','WH'),
}
ROAD_TAGS = list(TAG_ENDS)   # every road leg the model optimises
# NOTE: PCS_SPD_RAIL (far surplus->deficit pairs) travels by RAIL between rail heads; its two ROAD legs
#       (PC -> surplus RH, and deficit RH -> SP) are weighbridge-routed separately in the last section.

def coords(ntype, idx):
    df, la, lo, _, _ = NODETYPE[ntype]
    return float(df.at[idx, la]), float(df.at[idx, lo])
def node_id(ntype, idx):
    df, _, _, idc, nmc = NODETYPE[ntype]
    return df.at[idx, idc], df.at[idx, nmc]

In [ ]:
# ---------------- weighbridge routing (Section 4.3) ----------------
wb_lat = WB['WB_Lat'].to_numpy()
wb_lon = WB['WB_Long'].to_numpy()

def nearest_wb(lat, lon):
    d = haversine_km(lat, lon, wb_lat, wb_lon)
    k = int(np.argmin(d))
    return k, float(d[k])

def wb_route(plat, plon, qlat, qlon):
    '''Return the selected weighbridge and the two-leg distances for a road move p -> q.
       (wb_index, leg1 = d(p,wb*), leg2 = d(wb*,q), dtilde = leg1+leg2, direct = d(p,q), via).'''
    direct = float(haversine_km(plat, plon, qlat, qlon))
    b1, d_p_b1 = nearest_wb(plat, plon)                    # nearest to origin
    b2, d_q_b2 = nearest_wb(qlat, qlon)                    # nearest to destination
    d_b1_q = float(haversine_km(wb_lat[b1], wb_lon[b1], qlat, qlon))
    d_p_b2 = float(haversine_km(plat, plon, wb_lat[b2], wb_lon[b2]))
    L1 = d_p_b1 + d_b1_q                                   # route via origin's weighbridge
    L2 = d_p_b2 + d_q_b2                                   # route via destination's weighbridge
    if L1 <= L2:
        return b1, d_p_b1, d_b1_q, L1, direct, 'origin-WB'
    return b2, d_p_b2, d_q_b2, L2, direct, 'destination-WB'

In [ ]:
# ---------------- apply weighbridge routing to every road movement in the optimal solution ----------------
FLOWS = pd.read_csv('optimal_flows.csv')
road = FLOWS[FLOWS['tag'].isin(ROAD_TAGS)].copy()
print('road movements in the optimal solution:', len(road))

rec = []
for t in road.itertuples():
    o_t, d_t = TAG_ENDS[t.tag]
    plat, plon = coords(o_t, t.i); qlat, qlon = coords(d_t, t.j)
    wb, leg1, leg2, dtilde, direct, via = wb_route(plat, plon, qlat, qlon)
    oid, onm = node_id(o_t, t.i); did, dnm = node_id(d_t, t.j)
    rec.append({
        'tag': t.tag, 'origin_type': o_t, 'origin_id': oid, 'origin_name': onm,
        'dest_type': d_t, 'dest_id': did, 'dest_name': dnm, 'quantity': t.flow,
        'direct_km': direct, 'wb_id': WB.at[wb, 'WB_ID'], 'wb_name': WB.at[wb, 'WB_Name'],
        'wb_district': WB.at[wb, 'WB_District'], 'routed_via': via,
        'leg1_origin_to_wb_km': leg1, 'leg2_wb_to_dest_km': leg2,
        'wb_routed_km': dtilde, 'detour_km': dtilde - direct,
        'direct_quintal_km': t.flow * direct, 'wb_quintal_km': t.flow * dtilde,
    })
plan = pd.DataFrame(rec)
plan.head(10)

In [ ]:
# ---------------- leg splitting: each movement -> two legs (p -> wb*, wb* -> q), same quantity ----------------
legs = []
for p in plan.itertuples():
    legs.append({'tag': p.tag, 'leg': 'origin->WB', 'from': f'{p.origin_type}:{p.origin_id}',
                 'to': f'WB:{p.wb_id}', 'quantity': p.quantity, 'km': p.leg1_origin_to_wb_km})
    legs.append({'tag': p.tag, 'leg': 'WB->dest', 'from': f'WB:{p.wb_id}',
                 'to': f'{p.dest_type}:{p.dest_id}', 'quantity': p.quantity, 'km': p.leg2_wb_to_dest_km})
legs = pd.DataFrame(legs)
print('total realised road legs:', len(legs))
legs.head(6)

In [ ]:
# ---------------- summary: distance impact of weighbridge routing ----------------
dir_qkm = plan['direct_quintal_km'].sum()
wb_qkm  = plan['wb_quintal_km'].sum()
print(f"Road movements routed          : {len(plan):,}")
print(f"Direct quintal-km              : {dir_qkm:,.0f}")
print(f"Weighbridge-routed quintal-km  : {wb_qkm:,.0f}")
print(f"Extra (detour) quintal-km      : {wb_qkm - dir_qkm:,.0f}  ({100*(wb_qkm-dir_qkm)/dir_qkm:.1f}% over direct)")
print(f"Routed via origin WB / dest WB : {(plan.routed_via=='origin-WB').sum():,} / {(plan.routed_via=='destination-WB').sum():,}")

print("\nPer movement-type (tag):")
g = plan.groupby('tag').agg(moves=('quantity','size'), quantity=('quantity','sum'),
                            direct_qkm=('direct_quintal_km','sum'), wb_qkm=('wb_quintal_km','sum'))
g['detour_qkm'] = g['wb_qkm'] - g['direct_qkm']
print(g.round(0).to_string())

In [ ]:
# ---------------- busiest weighbridges (each movement weighs its quantity once at the chosen wb*) ----------------
wb_load = (plan.groupby('wb_id')['quantity'].sum()
               .sort_values(ascending=False).rename('quantity_weighed'))
busy = wb_load.reset_index().merge(WB[['WB_ID','WB_Name','WB_District','ss']],
                                   left_on='wb_id', right_on='WB_ID', how='left')
print('weighbridges used:', len(busy), 'of', len(WB))
print(busy.head(15)[['WB_ID','WB_Name','WB_District','ss','quantity_weighed']].to_string(index=False))

In [ ]:
# ---------------- save the realised movement plan and the leg list ----------------
plan.to_csv('weighbridge_plan.csv', index=False)
legs.to_csv('weighbridge_legs.csv', index=False)
print('wrote weighbridge_plan.csv (', len(plan), 'movements ) and weighbridge_legs.csv (', len(legs), 'legs )')

## Rail far-pairs (`PCS_SPD_RAIL`) — weighbridge routing of their road legs

The surplus→deficit *far pairs* move by **rail** between rail heads, but their two **road** legs are still
weighed: `PC → surplus rail head` and `deficit rail head → SP`. Using `railvia.csv` (which surplus rail head
each far pair uses) and `rail_shares.csv` (the fixed split of a surplus rail head to deficit rail heads),
each far-pair flow `f` on `(PC i → SP j)` is realised as:

- road `PC i → RH r` (quantity `f`) — weighbridge-routed
- rail `RH r → RH s` (quantity `f·share[r,s]`) — **no** weighbridge (rail)
- road `RH s → SP j` (quantity `f·share[r,s]`) — weighbridge-routed


In [ ]:
# ---------------- weighbridge routing for the rail far-pair ROAD legs ----------------
import os
if os.path.exists('railvia.csv') and os.path.exists('rail_shares.csv'):
    RV = pd.read_csv('railvia.csv')                 # i, j, r
    SH = pd.read_csv('rail_shares.csv')             # r, s, share
    rv = {(int(x.i), int(x.j)): int(x.r) for x in RV.itertuples()}
    dest_of = {}
    for x in SH.itertuples():
        dest_of.setdefault(int(x.r), []).append((int(x.s), float(x.share)))

    far = FLOWS[FLOWS['tag'] == 'PCS_SPD_RAIL']
    rail_legs = []
    for t in far.itertuples():
        i, j, f = int(t.i), int(t.j), t.flow
        r = rv[(i, j)]
        # road leg  PC i -> RH r
        plat, plon = coords('PC', i); rlat, rlon = coords('RH', r)
        wb, l1, l2, dt, dd, via = wb_route(plat, plon, rlat, rlon)
        rail_legs.append({'kind':'road PC->RH_S', 'from':f'PC:{node_id("PC",i)[0]}',
                          'to':f'RH:{node_id("RH",r)[0]}', 'quantity':f, 'wb_id':WB.at[wb,'WB_ID'],
                          'wb_routed_km':dt, 'direct_km':dd})
        # rail leg  RH r -> RH s  (no weighbridge) and road leg RH s -> SP j
        for s, sh in dest_of.get(r, []):
            qty = f * sh
            slat, slon = coords('RH', s); qlat, qlon = coords('SP', j)
            wb2, a1, a2, dt2, dd2, via2 = wb_route(slat, slon, qlat, qlon)
            rail_legs.append({'kind':'rail RH_S->RH_D', 'from':f'RH:{node_id("RH",r)[0]}',
                              'to':f'RH:{node_id("RH",s)[0]}', 'quantity':qty, 'wb_id':'(rail-no WB)',
                              'wb_routed_km':np.nan, 'direct_km':float(haversine_km(rlat,rlon,slat,slon))})
            rail_legs.append({'kind':'road RH_D->SP', 'from':f'RH:{node_id("RH",s)[0]}',
                              'to':f'SP:{node_id("SP",j)[0]}', 'quantity':qty, 'wb_id':WB.at[wb2,'WB_ID'],
                              'wb_routed_km':dt2, 'direct_km':dd2})
    rail_legs = pd.DataFrame(rail_legs)
    rail_legs.to_csv('weighbridge_rail_legs.csv', index=False)
    print('rail far-pair realised legs:', len(rail_legs))
    print(rail_legs.groupby('kind').agg(legs=('quantity','size'), quantity=('quantity','sum')).round(0).to_string())
else:
    print('railvia.csv / rail_shares.csv not found - skip rail far-pair road legs (export them from the model notebook).')